# Lab 2: Building a RAG-Enhanced Player Segmentation Agent

## Overview

In this lab, you'll build an AI-powered player segmentation agent using Strands framework, Amazon Bedrock, and the vector storage created in Lab 1. This agent uses Retrieval-Augmented Generation (RAG) to query player data contextually and discover meaningful segments.

### What You'll Build

**RAG-Enhanced Segmentation Agent** that:

- **Queries Vector Storage**: Uses semantic search to find relevant players
- **Contextual Analysis**: Retrieves player data based on behavioral patterns
- **Smart Segmentation**: Discovers segments using retrieved player contexts
- **Scalable Queries**: Works across all 4 player datasets from Lab 1

## Prerequisites

To execute this lab you will need:
* **Lab 1 completed** - Vector storage with player embeddings in S3
* AWS credentials configured
* Access to Amazon Bedrock Claude models

> **How to run:** Click each code cell and press Shift+Enter or the Run button. Execute cells in order from top to bottom.

## Step 1: Environment Setup

This section sets up all the dependencies and connections needed for our RAG-enhanced segmentation agent. We're installing the Strands framework for building AI agents, connecting to AWS services, and preparing the same embedding model used in Lab 1 to ensure consistency with our vector storage.

In [ ]:
!pip install strands-agents matplotlib -q

### Import Libraries and Configure AWS Clients

Import required libraries and establish connections to AWS services using the same configuration from Lab 1.

In [ ]:
import boto3
import json
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime
from strands import Agent, tool

# Vector Storage Configuration - connect to the S3 bucket created in Lab 1
VECTOR_BUCKET = "game-embeddings"
AWS_REGION = "us-west-2"

# Initialize AWS clients
s3 = boto3.client('s3', region_name=AWS_REGION)
s3vectors = boto3.client('s3vectors', region_name=AWS_REGION)
bedrock = boto3.client('bedrock-runtime', region_name=AWS_REGION)

print("✅ Setup complete")

## Step 2: Create Vector Query Tools

This section creates Python functions decorated with @tool that extend the AI agent's capabilities beyond simple text generation. Tools are the primary mechanism for enabling agents to interact with external systems and access data. Learn more about [Strands tools and how they work](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/tools/tools_overview/). These tools enable Retrieval-Augmented Generation (RAG) - a technique that enhances AI responses by retrieving relevant information from external data sources. Learn more about [RAG and its applications](https://aws.amazon.com/what-is/retrieval-augmented-generation/).

The first tool enables the agent to perform semantic search across the S3 Vectors indices from Lab 1 using get_vectors API based on a natural language query.

In [ ]:
@tool
def query_player_segments(query: str, dataset: str = "player_profiles") -> str:
    """Query player data using S3 Vectors semantic search.
    
    Args:
        query: Natural language query (e.g., 'high spending players', 'at-risk players')
        dataset: Dataset to query ('player_profiles', 'lifetime_spend', 'session_history', 'player_attributes')
        
    Returns:
        str: JSON string of matching player data
    """
    print(f"🔍 query_player_segments called with: query='{query}', dataset='{dataset}'")
    
    try:
        response = bedrock.invoke_model(
            modelId='cohere.embed-english-v3',
            body=json.dumps({
                "texts": [query],
                "input_type": "search_query"
            })
        ) 
        model_response = json.loads(response["body"].read())
        query_embedding = model_response["embeddings"][0]
        
        # Use S3 Vectors API for semantic search
        response = s3vectors.query_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=dataset.replace('_', '-'),
            queryVector={'float32': query_embedding},
            topK=30,
            returnMetadata=True,
            returnDistance=True
        )
        
        # Format results for the agent
        results = []
        for vector in response.get('vectors', []):
            results.append({
                'key': vector.get('key'),
                'data': vector.get('metadata', {})
            })
        
        if results:
            sample_player = results[0]['data']
            sample_snippet = {k: v for k, v in list(sample_player.items())}
            print(f"Found {len(results)} players. Sample: {sample_snippet}\n\n")
        else:
            print("Found 0 matching players")

        return json.dumps(results, indent=2)
    
    except Exception as e:
        print(f"❌ Query failed: {str(e)}")
        return json.dumps({"error": f"Query failed: {str(e)}"})
    
print("✅ Query player segments tool created")

The second tool retrieves comprehensive player data across all datasets using S3 Vectors get_vectors API based on a list of player IDs. This enables the agent to build complete player profiles by gathering context from multiple data sources on the requested players.

In [ ]:
@tool
def get_player_context(player_ids: list, datasets: list = None) -> str:
    """Get comprehensive context for specific players using S3 Vectors.
    
    Args:
        player_ids: List of player IDs to get context for
        datasets: List of datasets to search (default: all)
        
    Returns:
        str: JSON string of player context from multiple datasets
    """
    print(f"🔍 get_player_context called with: player_ids={player_ids}, datasets={datasets}")
    
    if datasets is None:
        datasets = ["player_profiles", "lifetime_spend", "session_history", "player_attributes"]
    
    player_context = {}
    
    # Search each dataset for the specified player IDs using S3 Vectors
    for dataset in datasets:
        print(f"📊 Querying dataset: {dataset}")
        try:
            # Use S3 Vectors get_vectors API to retrieve specific player data
            response = s3vectors.get_vectors(
                vectorBucketName=VECTOR_BUCKET,
                indexName=dataset.replace('_', '-'),
                keys=player_ids,  # Get vectors for specific player IDs
                returnMetadata=True
            )
            
            # Build comprehensive player profiles from the response
            for vector in response.get('vectors', []):
                player_id = vector.get('key')
                if player_id in player_ids:
                    if player_id not in player_context:
                        player_context[player_id] = {}
                    player_context[player_id][dataset] = vector.get('metadata', {})
        except Exception as e:
            print(f"⚠️ Failed to query {dataset}: {str(e)}")
            continue
    
    print(f"✅ Retrieved context for {len(player_context)} players across {len(datasets)} datasets")
    return json.dumps(player_context, indent=2)

print("✅ Get player context tool created")

## Step 3: Create Visualization Tool

This tool creates charts and graphs to help visualize player data and support segment analysis. The AI agent can call this function to generate visual evidence for its segmentation insights, making the analysis more compelling and easier to understand for business stakeholders.

In [ ]:
@tool
def create_visualization(chart_type: str, x_column: str, y_column: str = None, title: str = "Player Analysis", dataset: str = "player_profiles") -> str:
    """Create visualizations from player data.
    
    Args:
        chart_type: 'histogram', 'scatter', 'bar'
        x_column: Column name for x-axis
        y_column: Column name for y-axis (for scatter/bar)
        title: Chart title
        dataset: Which dataset to use ('player_profiles', 'lifetime_spend', 'session_history', 'player_attributes')
    """
    try:
        from io import BytesIO
        
        # Get bucket from SSM
        ssm = boto3.client('ssm', region_name=AWS_REGION)
        bucket_name = ssm.get_parameter(Name='/workshop/games-data-bucket-name')['Parameter']['Value']
        
        # Load the specific dataset the agent requested
        response = s3.get_object(Bucket=bucket_name, Key=f"{dataset}.json")
        data = json.loads(response['Body'].read().decode('utf-8'))
        df = pd.DataFrame(data)
        
        if df.empty or x_column not in df.columns:
            return f"Error: Column '{x_column}' not found in {dataset}"
        
        plt.figure(figsize=(10, 6))
        
        if chart_type == "histogram":
            # Use binned aggregation for large datasets
            plt.hist(df[x_column].dropna(), bins=15, alpha=0.7)
            plt.xlabel(x_column)
            plt.ylabel("Frequency")
        elif chart_type == "scatter" and y_column and y_column in df.columns:
            # Create aggregated scatter plot using groupby means
            if df[x_column].dtype == 'object' or df[y_column].dtype == 'object':
                # For categorical data, use value counts
                grouped = df.groupby([x_column, y_column]).size().reset_index(name='count')
                plt.scatter(grouped[x_column], grouped[y_column], s=grouped['count']*10, alpha=0.6)
            else:
                # For numerical data, bin and average
                df_clean = df[[x_column, y_column]].dropna()
                x_bins = pd.cut(df_clean[x_column], bins=20)
                grouped = df_clean.groupby(x_bins)[y_column].mean()
                plt.scatter(range(len(grouped)), grouped.values, alpha=0.6)
                plt.xticks(range(len(grouped)), [f"{int(x.left)}-{int(x.right)}" for x in grouped.index], rotation=45)
            plt.xlabel(x_column)
            plt.ylabel(y_column)
        elif chart_type == "bar":
            # Use full dataset for counts - no sampling
            value_counts = df[x_column].value_counts()
            plt.bar(range(len(value_counts)), value_counts.values)
            plt.xticks(range(len(value_counts)), value_counts.index, rotation=45)
            plt.ylabel("Count")
        
        plt.title(title)
        plt.tight_layout()
        
        # Save to S3 instead of local file
        buffer = BytesIO()
        plt.savefig(buffer, format='png', dpi=150, bbox_inches='tight')
        buffer.seek(0)
        
        # Upload to S3
        s3_key = f"visualizations/chart_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        s3.put_object(
            Bucket=bucket_name,
            Key=s3_key,
            Body=buffer.getvalue(),
            ContentType='image/png'
        )
        
        plt.close()
        
        # Return S3 URL with summary stats
        total_records = len(df)
        s3_url = f"https://{bucket_name}.s3.{AWS_REGION}.amazonaws.com/{s3_key}"
        return f"Visualization created from {total_records} total records: {s3_url}"
        
    except Exception as e:
        return f"Visualization error: {str(e)}"

print("✅ Create visualization tool created")

## Step 4: Create RAG-Enhanced Segmentation Agent

Now we provide all our tools to an intelligent AI agent using Claude Sonnet as the reasoning engine. AI agents are software programs that can interact with their environment, collect data, and use that data to perform self-directed tasks to meet predetermined goals. Learn more about [what AI agents are](https://aws.amazon.com/what-is/ai-agents/).

We're using [Strands Agents](https://github.com/strands-agents/sdk-python) - a simple yet powerful SDK that takes a model-driven approach to building AI agents in just a few lines of code. The system prompt defines the agent's role as a player segmentation expert and provides clear instructions on using the RAG workflow to discover actionable player segments through systematic data exploration.

In [ ]:
def create_segmentation_agent():
  """Create the RAG-enhanced player segmentation agent."""
  
  return Agent(
    model="global.anthropic.claude-sonnet-5-5",
    tools=[query_player_segments, get_player_context, create_visualization],
    system_prompt=f"""You are an advanced player segmentation analyst with access to RAG-enhanced gaming data.

    Your role is to discover actionable player segments using semantic search across comprehensive datasets:
    - player_profiles: Core engagement and demographic data
    - lifetime_spend: Purchase history and spending patterns
    - session_history: Behavioral and activity patterns
    - player_attributes: Gameplay context and progression
    
    EFFICIENT EXPLORATION APPROACH:
    1. Run exactly 4 targeted queries (one per dataset) to understand key patterns
    2. Validate query results show different players - if same players appear, adjust approach
    3. Create 3-4 segments using broad, data-driven criteria
    4. Focus on patterns that appear in 10%+ of results
    
    MANDATORY DATA EXPLORATION:
    - Query player_profiles: Explore engagement and churn patterns to find natural breakpoints
    - Query lifetime_spend: Discover spending tiers and transaction frequency patterns  
    - Query session_history: Identify session length and frequency behavioral clusters
    - Query player_attributes: Find common gameplay role and skill combinations
    
    DISCOVERY APPROACH:
    - Start with exploratory queries to understand data ranges
    - Look for natural clusters in the results (high/medium/low groupings)
    - Use actual data patterns you observe to define segments
    - Don't assume thresholds - let the data guide segment boundaries
        
    QUERY VALIDATION RULES:
    - If multiple queries return identical players, investigate why
    - Verify result counts vary based on query specificity
    - Check that each dataset returns different player patterns
    - If seeing repeated samples, broaden your query terms
    
    SEGMENT SIZE REQUIREMENTS:
    - Create exactly 3-4 segments
    - Each segment must realistically capture 10-25% of any player base
    - Avoid overly specific combinations (don't stack 3+ criteria)
    - If a pattern shows <30 players in sample, make it broader
    
    SEGMENT VALIDATION:
    - Include actual player counts from your queries in descriptions
    - Base recommendations on specific data patterns you discovered
    - Reference actual player behaviors found in the results
    - Avoid generic business templates - use your actual findings
    
    AVOID TINY SEGMENTS:
    - Don't combine platform + faction + spending + engagement
    - If you see a pattern in only a few players, generalize it
    - Think "would this segment exist in a 10,000 player dataset?"
    - Prioritize broad behavioral patterns over specific combinations
    
    OUTPUT FORMAT:
    Always return segments in this JSON structure:
    {{
    "segments": [
        {{
        "name": "Segment Name",
        "description": "Data-driven description with actual findings",
        "criteria": "SELECT player_id FROM player_profiles p JOIN lifetime_spend s ON p.player_id = s.player_id WHERE condition",
        "business_value": "Strategy based on discovered patterns",
        }}
    ]
    }}
    
    IMPORTANT: The 'criteria' field must contain executable SQL using exact field names:
    - player_id, churn_risk_score, engagement_intensity_score, faction_allegiance, geographic_region, platform, total_spend_usd, total_transactions
    
    Focus on segments that drive business value while being broad enough to be actionable at scale."""
  )

# Create the agent
agent = create_segmentation_agent()
print("✅ RAG-Enhanced Segmentation Agent created")

## Step 5: Test the Agent

This section tests our RAG-enhanced agent with a comprehensive segmentation scenario to verify that it can effectively use the vector search tools to discover meaningful player segments. The test validates the complete workflow from semantic search to actionable business insights.

In [ ]:
response = agent("Discover actionable player segments for our game LiveOps strategy.")

print("🤖 Agent Response:")
print(response)

Now let's test the agent's ability to analyze individual players by asking about a specific player. This will demonstrate how the agent uses the `get_player_context` tool to retrieve comprehensive player data across all datasets and provide detailed insights.

In [ ]:
response = agent("Tell me everything about player_000558. What kind of player are they and what segment would they fit into?")

print("🤖 Agent Response:")
print(response)

## Step 6: Save Agent Code

This section saves the complete agent code to a file for use in Lab 3 deployment. The saved agent includes all tools, configurations, and system prompts needed for production deployment to Amazon Bedrock AgentCore Runtime.

In [ ]:
%%writefile player_segmentation_agent.py
import boto3
import json
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime
from strands import Agent, tool

# Configuration
VECTOR_BUCKET = "game-embeddings"
AWS_REGION = "us-west-2"
s3 = boto3.client('s3', region_name=AWS_REGION)
s3vectors = boto3.client('s3vectors', region_name=AWS_REGION)
bedrock = boto3.client('bedrock-runtime', region_name=AWS_REGION)

@tool
def query_player_segments(query: str, dataset: str = "player_profiles") -> str:
    """Query player data using S3 Vectors semantic search.
    
    Args:
        query: Natural language query (e.g., 'high spending players', 'at-risk players')
        dataset: Dataset to query ('player_profiles', 'lifetime_spend', 'session_history', 'player_attributes')
        
    Returns:
        str: JSON string of matching player data
    """
    print(f"🔍 query_player_segments called with: query='{query}', dataset='{dataset}'")
    
    try:
        response = bedrock.invoke_model(
            modelId='cohere.embed-english-v3',
            body=json.dumps({
                "texts": [query],
                "input_type": "search_query"
            })
        ) 
        model_response = json.loads(response["body"].read())
        query_embedding = model_response["embeddings"][0]
        
        # Use S3 Vectors API for semantic search
        response = s3vectors.query_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=dataset.replace('_', '-'),
            queryVector={'float32': query_embedding},
            topK=30,
            returnMetadata=True,
            returnDistance=True
        )
        
        # Format results for the agent
        results = []
        for vector in response.get('vectors', []):
            results.append({
                'key': vector.get('key'),
                'data': vector.get('metadata', {})
            })
        
        if results:
            sample_player = results[0]['data']
            sample_snippet = {k: v for k, v in list(sample_player.items())}
            print(f"Found {len(results)} players. Sample: {sample_snippet}\n\n")
        else:
            print("Found 0 matching players")

        return json.dumps(results, indent=2)
    
    except Exception as e:
        print(f"❌ Query failed: {str(e)}")
        return json.dumps({"error": f"Query failed: {str(e)}"})

@tool
def get_player_context(player_ids: list, datasets: list = None) -> str:
    """Get comprehensive context for specific players using S3 Vectors.
    
    Args:
        player_ids: List of player IDs to get context for
        datasets: List of datasets to search (default: all)
        
    Returns:
        str: JSON string of player context from multiple datasets
    """
    print(f"🔍 get_player_context called with: player_ids={player_ids}, datasets={datasets}")
    
    if datasets is None:
        datasets = ["player_profiles", "lifetime_spend", "session_history", "player_attributes"]
    
    player_context = {}
    
    # Search each dataset for the specified player IDs using S3 Vectors
    for dataset in datasets:
        print(f"📊 Querying dataset: {dataset}")
        try:
            # Use S3 Vectors get_vectors API to retrieve specific player data
            response = s3vectors.get_vectors(
                vectorBucketName=VECTOR_BUCKET,
                indexName=dataset.replace('_', '-'),
                keys=player_ids,  # Get vectors for specific player IDs
                returnMetadata=True
            )
            
            # Build comprehensive player profiles from the response
            for vector in response.get('vectors', []):
                player_id = vector.get('key')
                if player_id in player_ids:
                    if player_id not in player_context:
                        player_context[player_id] = {}
                    player_context[player_id][dataset] = vector.get('metadata', {})
        except Exception as e:
            print(f"⚠️ Failed to query {dataset}: {str(e)}")
            continue
    
    print(f"✅ Retrieved context for {len(player_context)} players across {len(datasets)} datasets")
    return json.dumps(player_context, indent=2)

@tool
def create_visualization(chart_type: str, x_column: str, y_column: str = None, title: str = "Player Analysis", dataset: str = "player_profiles") -> str:
    """Create visualizations from player data.
    
    Args:
        chart_type: 'histogram', 'scatter', 'bar'
        x_column: Column name for x-axis
        y_column: Column name for y-axis (for scatter/bar)
        title: Chart title
        dataset: Which dataset to use ('player_profiles', 'lifetime_spend', 'session_history', 'player_attributes')
    """
    try:
        from io import BytesIO
        
        # Get bucket from SSM
        ssm = boto3.client('ssm', region_name=AWS_REGION)
        bucket_name = ssm.get_parameter(Name='/workshop/games-data-bucket-name')['Parameter']['Value']
        
        # Load the specific dataset the agent requested
        response = s3.get_object(Bucket=bucket_name, Key=f"{dataset}.json")
        data = json.loads(response['Body'].read().decode('utf-8'))
        df = pd.DataFrame(data)
        
        if df.empty or x_column not in df.columns:
            return f"Error: Column '{x_column}' not found in {dataset}"
        
        plt.figure(figsize=(10, 6))
        
        if chart_type == "histogram":
            # Use binned aggregation for large datasets
            plt.hist(df[x_column].dropna(), bins=15, alpha=0.7)
            plt.xlabel(x_column)
            plt.ylabel("Frequency")
        elif chart_type == "scatter" and y_column and y_column in df.columns:
            # Create aggregated scatter plot using groupby means
            if df[x_column].dtype == 'object' or df[y_column].dtype == 'object':
                # For categorical data, use value counts
                grouped = df.groupby([x_column, y_column]).size().reset_index(name='count')
                plt.scatter(grouped[x_column], grouped[y_column], s=grouped['count']*10, alpha=0.6)
            else:
                # For numerical data, bin and average
                df_clean = df[[x_column, y_column]].dropna()
                x_bins = pd.cut(df_clean[x_column], bins=20)
                grouped = df_clean.groupby(x_bins)[y_column].mean()
                plt.scatter(range(len(grouped)), grouped.values, alpha=0.6)
                plt.xticks(range(len(grouped)), [f"{int(x.left)}-{int(x.right)}" for x in grouped.index], rotation=45)
            plt.xlabel(x_column)
            plt.ylabel(y_column)
        elif chart_type == "bar":
            # Use full dataset for counts - no sampling
            value_counts = df[x_column].value_counts()
            plt.bar(range(len(value_counts)), value_counts.values)
            plt.xticks(range(len(value_counts)), value_counts.index, rotation=45)
            plt.ylabel("Count")
        
        plt.title(title)
        plt.tight_layout()
        
        # Save to S3 instead of local file
        buffer = BytesIO()
        plt.savefig(buffer, format='png', dpi=150, bbox_inches='tight')
        buffer.seek(0)
        
        # Upload to S3
        s3_key = f"visualizations/chart_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        s3.put_object(
            Bucket=bucket_name,
            Key=s3_key,
            Body=buffer.getvalue(),
            ContentType='image/png'
        )
        
        plt.close()
        
        # Return S3 URL with summary stats
        total_records = len(df)
        s3_url = f"https://{bucket_name}.s3.{AWS_REGION}.amazonaws.com/{s3_key}"
        return f"Visualization created from {total_records} total records: {s3_url}"
        
    except Exception as e:
        return f"Visualization error: {str(e)}"

def create_segmentation_agent():
  """Create the RAG-enhanced player segmentation agent."""
  
  return Agent(
    model="global.anthropic.claude-sonnet-5-5",
    tools=[query_player_segments, get_player_context, create_visualization],
    system_prompt=f"""You are an advanced player segmentation analyst with access to RAG-enhanced gaming data.

    Your role is to discover actionable player segments using semantic search across comprehensive datasets:
    - player_profiles: Core engagement and demographic data
    - lifetime_spend: Purchase history and spending patterns
    - session_history: Behavioral and activity patterns
    - player_attributes: Gameplay context and progression
    
    EFFICIENT EXPLORATION APPROACH:
    1. Run exactly 4 targeted queries (one per dataset) to understand key patterns
    2. Validate query results show different players - if same players appear, adjust approach
    3. Create 3-4 segments using broad, data-driven criteria
    4. Focus on patterns that appear in 10%+ of results
    
    MANDATORY DATA EXPLORATION:
    - Query player_profiles: Explore engagement and churn patterns to find natural breakpoints
    - Query lifetime_spend: Discover spending tiers and transaction frequency patterns  
    - Query session_history: Identify session length and frequency behavioral clusters
    - Query player_attributes: Find common gameplay role and skill combinations
    
    DISCOVERY APPROACH:
    - Start with exploratory queries to understand data ranges
    - Look for natural clusters in the results (high/medium/low groupings)
    - Use actual data patterns you observe to define segments
    - Don't assume thresholds - let the data guide segment boundaries
        
    QUERY VALIDATION RULES:
    - If multiple queries return identical players, investigate why
    - Verify result counts vary based on query specificity
    - Check that each dataset returns different player patterns
    - If seeing repeated samples, broaden your query terms
    
    SEGMENT SIZE REQUIREMENTS:
    - Create exactly 3-4 segments
    - Each segment must realistically capture 10-25% of any player base
    - Avoid overly specific combinations (don't stack 3+ criteria)
    - If a pattern shows <30 players in sample, make it broader
    
    SEGMENT VALIDATION:
    - Include actual player counts from your queries in descriptions
    - Base recommendations on specific data patterns you discovered
    - Reference actual player behaviors found in the results
    - Avoid generic business templates - use your actual findings
    
    AVOID TINY SEGMENTS:
    - Don't combine platform + faction + spending + engagement
    - If you see a pattern in only a few players, generalize it
    - Think "would this segment exist in a 10,000 player dataset?"
    - Prioritize broad behavioral patterns over specific combinations
    
    OUTPUT FORMAT:
    Always return segments in this JSON structure:
    {{
    "segments": [
        {{
        "name": "Segment Name",
        "description": "Data-driven description with actual findings",
        "criteria": "SELECT player_id FROM player_profiles p JOIN lifetime_spend s ON p.player_id = s.player_id WHERE condition",
        "business_value": "Strategy based on discovered patterns",
        }}
    ]
    }}
    
    IMPORTANT: The 'criteria' field must contain executable SQL using exact field names:
    - player_id, churn_risk_score, engagement_intensity_score, faction_allegiance, geographic_region, platform, total_spend_usd, total_transactions
    
    Focus on segments that drive business value while being broad enough to be actionable at scale."""
  )

agent = create_segmentation_agent()

## Congratulations! 🎉

You have successfully:

✅ **Built RAG-enhanced agent** with S3 Vectors integration from Lab 1  
✅ **Created semantic search tools** for contextual player queries across datasets  
✅ **Implemented multi-dataset context** retrieval for comprehensive player profiles  
✅ **Tested agent capabilities** with various segmentation scenarios  
✅ **Saved agent code** for deployment in Lab 3

### Key Tools Created:
- **query_player_vectors**: Semantic search across player datasets
- **get_player_context**: Multi-dataset context retrieval for specific players
- **create_visualization**: Chart generation for segment analysis

### Key Concepts Learned:
- **RAG Architecture**: Combining retrieval with generation for enhanced AI responses
- **Vector Search**: Using embeddings to find semantically similar player data
- **Multi-Dataset Queries**: Gathering comprehensive context across data sources
- **Agent Tools**: Building specialized functions for AI agent capabilities

This RAG-enhanced agent provides the foundation for production deployment in Lab 3.